# 03_baseline_models

**Objective:** Classical ML baseline models




# 03 — Baseline Model Experiments

## 03.1 — Setup and Reproducibility

**Objective:** Establish baseline performance and identify
promising model families for loan default prediction.

- Train: January 2013 – September 2015
- Validation: October 2015 – December 2015
- Final holdout: January – December 2016
- Target: Fully Paid = 0, Charged Off = 1

Use the feature manifest from Notebook 02.
Do not use 2016 outcomes for modeling decisions.

Primary metrics: PR-AUC, ROC-AUC, default precision,
recall, F1, balanced accuracy and confusion matrix.


In [1]:

from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SOURCE = ROOT / "data/interim/lendingclub_2013_2016_36m_raw.parquet"
MANIFEST_PATH = ROOT / "reports/preprocessing/02_feature_manifest_draft.csv"

REPORT_DIR = ROOT / "reports/modeling"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists()
assert MANIFEST_PATH.exists()

# Feature decisions established in Notebook 02
decisions = pd.read_csv(MANIFEST_PATH)

numeric_features = decisions.loc[
    decisions["model_status"] == "CANDIDATE_NUMERIC",
    "feature"
].tolist()

categorical_features = decisions.loc[
    decisions["model_status"].isin([
        "APPROVED_CATEGORICAL",
        "REVIEW_BASELINE_CONTRIBUTION"
    ]),
    "feature"
].tolist()

assert len(numeric_features) == 50
assert set(categorical_features) == {"home_ownership", "addr_state"}
assert decisions.loc[
    decisions["feature"] == "emp_length", "model_status"
].iloc[0] == "ENGINEER_NUMERIC"

# Recreate the frozen chronological split
lf = pl.scan_parquet(SOURCE)

train_lf = lf.filter(
    (pl.col("issue_date") >= date(2013, 1, 1)) &
    (pl.col("issue_date") < date(2015, 10, 1))
)

val_lf = lf.filter(
    (pl.col("issue_date") >= date(2015, 10, 1)) &
    (pl.col("issue_date") < date(2016, 1, 1))
)

test_lf = lf.filter(
    (pl.col("issue_date") >= date(2016, 1, 1)) &
    (pl.col("issue_date") < date(2017, 1, 1))
)

# Targets from development data only
y_train = train_lf.select("target").collect().to_numpy().ravel()
y_val = val_lf.select("target").collect().to_numpy().ravel()

assert len(y_train) == 457355
assert len(y_val) == 88810
assert set(np.unique(y_train)) == {0, 1}
assert set(np.unique(y_val)) == {0, 1}

print("Train:", len(y_train))
print("Validation:", len(y_val))
print("Test: reserved (2016)")

print("\nNumerical candidates:", len(numeric_features))
print("Categorical:", categorical_features)
print("Setup: PASS")


Train: 457355
Validation: 88810
Test: reserved (2016)

Numerical candidates: 50
Categorical: ['home_ownership', 'addr_state']
Setup: PASS



## 03.2 — Dummy Classifier Baseline

**Hypothesis:** A model using borrower information should
outperform a classifier that only knows class prevalence.

Train a DummyClassifier with the training class prior.
Evaluate on the validation period.

Record all metrics for comparison with later models.
Accuracy alone is not sufficient due to class imbalance.


In [2]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# Dummy requires only a placeholder feature
X_dummy_train = np.zeros((len(y_train), 1), dtype=np.float32)
X_dummy_val = np.zeros((len(y_val), 1), dtype=np.float32)

dummy = DummyClassifier(strategy="prior")
dummy.fit(X_dummy_train, y_train)

y_pred = dummy.predict(X_dummy_val)
y_prob = dummy.predict_proba(X_dummy_val)[
    :, list(dummy.classes_).index(1)
]

dummy_result = {
    "model": "Dummy (prior)",
    "configuration": "No borrower features",
    "accuracy": accuracy_score(y_val, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_val, y_pred),
    "precision_default": precision_score(
        y_val, y_pred, zero_division=0
    ),
    "recall_default": recall_score(y_val, y_pred),
    "f1_default": f1_score(y_val, y_pred),
    "roc_auc": roc_auc_score(y_val, y_prob),
    "pr_auc": average_precision_score(y_val, y_prob),
}

# Initialize comparison record for upcoming models
results = [dummy_result]

results_df = pd.DataFrame(results)
display(results_df.round(4))

print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, y_pred, labels=[0, 1]))

results_df.to_csv(
    REPORT_DIR / "03_baseline_comparison.csv",
    index=False
)

print("\nBaseline results saved.")


,model,configuration,accuracy,balanced_accuracy,precision_default,recall_default,f1_default,roc_auc,pr_auc
0,Dummy (prior),No borrower features,0.8516,0.5,0.0,0.0,0.0,0.5,0.1484



Confusion Matrix [[TN, FP], [FN, TP]]:
[[75627     0]
 [13183     0]]

Baseline results saved.



## 03.3 — Logistic Regression Baseline

**Hypothesis:** Borrower and credit-history features contain
predictive information beyond the overall default rate.

Model:
- Logistic Regression with L2 regularization.
- Median imputation and standardization for numerical data.
- One-hot encoding for categorical variables.
- Three missingness indicators.
- Default probability threshold = 0.50.

Train on 2013–September 2015.
Evaluate on October–December 2015.

Compare PR-AUC, ROC-AUC, default precision, recall,
F1, balanced accuracy and confusion matrix against Dummy.

No PCA, engineered features or class balancing yet.


In [4]:

# Cell 3 — Recreate baseline feature preparation

numeric_input_features = numeric_features + ["emp_length_years"]

indicator_features = [
    "delinq_recency_missing",
    "emp_length_missing",
    "recent_inq_missing",
]

input_features = (
    numeric_input_features
    + indicator_features
    + categorical_features
)

def prepare_features(data):
    expressions = []

    for c in numeric_features:
        raw = pl.col(c).cast(pl.String).str.strip_chars()

        if c == "revol_util":
            raw = raw.str.replace_all("%", "")

        value = raw.cast(pl.Float32, strict=False)

        if c == "dti":
            value = (
                pl.when(value.is_finite().fill_null(False) & (value >= 0))
                .then(value)
                .otherwise(None)
            )

        expressions.append(value.alias(c))

    # Ordinal employment-length conversion
    emp = pl.col("emp_length").cast(pl.String).str.strip_chars()

    emp_years = (
        pl.when(emp == "< 1 year").then(0)
        .when(emp == "10+ years").then(10)
        .otherwise(
            emp.str.extract(r"^(\d+)\s+years?$", 1)
            .cast(pl.Int32, strict=False)
        )
    )

    expressions.append(emp_years.alias("emp_length_years"))

    for c in categorical_features:
        expressions.append(
            pl.col(c).cast(pl.String).str.strip_chars().alias(c)
        )

    expressions.extend(["target", "issue_date"])

    prepared = data.select(expressions)

    # Indicators must be calculated AFTER numeric conversion
    prepared = prepared.with_columns(
        pl.col("mths_since_last_delinq")
        .is_null().cast(pl.Int8)
        .alias("delinq_recency_missing"),

        pl.col("emp_length_years")
        .is_null().cast(pl.Int8)
        .alias("emp_length_missing"),

        pl.col("mths_since_recent_inq")
        .is_null().cast(pl.Int8)
        .alias("recent_inq_missing"),
    )

    return prepared


train_prepared_lf = prepare_features(train_lf)
val_prepared_lf = prepare_features(val_lf)

X_train = train_prepared_lf.select(
    input_features
).collect().to_pandas()

X_val = val_prepared_lf.select(
    input_features
).collect().to_pandas()

assert X_train.shape == (457355, 56)
assert X_val.shape == (88810, 56)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Feature preparation: PASS")


Training: (457355, 56)
Validation: (88810, 56)
Feature preparation: PASS


In [5]:

# 03.3 — Logistic Regression Baseline (Corrected)

import time
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    FunctionTransformer,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

# --------------------------------------------------
# 1. Categorical normalization
# --------------------------------------------------

def normalize_categories(X):
    X = X.copy()

    for c in X.columns:
        X[c] = X[c].astype("string").str.strip()
        X[c] = X[c].replace({
            "": pd.NA,
            "nan": pd.NA,
            "null": pd.NA,
        })
        X[c] = X[c].astype(object).where(
            X[c].notna(), np.nan
        )

    return X


# --------------------------------------------------
# 2. Preprocessing pipelines
# --------------------------------------------------

numeric_pipeline_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    (
        "normalize",
        FunctionTransformer(
            normalize_categories,
            validate=False,
            feature_names_out="one-to-one",
        ),
    ),
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="MISSING",
        ),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            dtype=np.float32,
        ),
    ),
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline_lr,
            numeric_input_features,
        ),
        (
            "indicators",
            "passthrough",
            indicator_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)


# --------------------------------------------------
# 3. Logistic Regression model
# --------------------------------------------------

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor_lr),
    (
        "classifier",
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=300,
            tol=1e-3,
            class_weight=None,
            random_state=42,
        ),
    ),
])


# --------------------------------------------------
# 4. Train on training data only
# --------------------------------------------------

start = time.perf_counter()

lr_pipeline.fit(X_train, y_train)

fit_seconds = time.perf_counter() - start


# --------------------------------------------------
# 5. Validation predictions
# --------------------------------------------------

lr_pred = lr_pipeline.predict(X_val)
lr_prob = lr_pipeline.predict_proba(X_val)[:, 1]


# --------------------------------------------------
# 6. Evaluation
# --------------------------------------------------

lr_result = {
    "model": "Logistic Regression",
    "configuration": "Original features; L2; unweighted",
    "accuracy": accuracy_score(y_val, lr_pred),
    "balanced_accuracy": balanced_accuracy_score(y_val, lr_pred),
    "precision_default": precision_score(
        y_val, lr_pred, zero_division=0
    ),
    "recall_default": recall_score(
        y_val, lr_pred, zero_division=0
    ),
    "f1_default": f1_score(
        y_val, lr_pred, zero_division=0
    ),
    "roc_auc": roc_auc_score(y_val, lr_prob),
    "pr_auc": average_precision_score(y_val, lr_prob),
}


# --------------------------------------------------
# 7. Update model comparison
# --------------------------------------------------

# Keep Dummy and any other previously saved results
results = [
    r for r in results
    if r["model"] != "Logistic Regression"
]

results.append(lr_result)

results_df = pd.DataFrame(results)

print("MODEL COMPARISON")
display(results_df.round(4))


# --------------------------------------------------
# 8. Confusion matrix and classification report
# --------------------------------------------------

print("\nLOGISTIC REGRESSION CONFUSION MATRIX")
print(confusion_matrix(y_val, lr_pred, labels=[0, 1]))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_val,
        lr_pred,
        target_names=["Fully Paid", "Charged Off"],
        zero_division=0,
    )
)


# --------------------------------------------------
# 9. Model information
# --------------------------------------------------

n_features = lr_pipeline.named_steps[
    "classifier"
].n_features_in_

print("\nMODEL INFORMATION")
print("Transformed features:", n_features)
print("Training time:", round(fit_seconds, 2), "seconds")
print("Iterations:", lr_pipeline.named_steps[
    "classifier"
].n_iter_[0])

assert n_features == 109
assert np.isfinite(lr_prob).all()


# --------------------------------------------------
# 10. Save comparison
# --------------------------------------------------

REPORT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    REPORT_DIR / "03_baseline_comparison.csv",
    index=False,
)

print("\nResults saved successfully.")


c:\Users\kau_shal\miniconda3\envs\ml\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


MODEL COMPARISON


,model,configuration,accuracy,balanced_accuracy,precision_default,recall_default,f1_default,roc_auc,pr_auc
0,Dummy (prior),No borrower features,0.8516,0.5000,0.0000,0.0000,0.0000,0.5000,0.1484
1,Logistic Regression,Original features; L2; unweighted,0.8514,0.5013,0.4259,0.0035,0.0069,0.6828,0.2642



LOGISTIC REGRESSION CONFUSION MATRIX
[[75565    62]
 [13137    46]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Fully Paid       0.85      1.00      0.92     75627
 Charged Off       0.43      0.00      0.01     13183

    accuracy                           0.85     88810
   macro avg       0.64      0.50      0.46     88810
weighted avg       0.79      0.85      0.78     88810


MODEL INFORMATION
Transformed features: 109
Training time: 4.31 seconds
Iterations: 14

Results saved successfully.



## 03.4 — Logistic Regression Threshold Investigation

The unweighted Logistic Regression achieves ROC-AUC 0.6828
and PR-AUC 0.2642, but detects very few defaults at the
default classification threshold of 0.50.

Investigate lower decision thresholds on validation data.

Compare precision, recall, F1, and balanced accuracy.
Identify the threshold maximizing validation F1.

Treat the selected threshold as provisional; do not use
the 2016 final holdout for threshold selection.


In [6]:

# Cell 5 — Logistic Regression threshold investigation
# Reuses lr_prob and y_val — no retraining

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
)

thresholds_to_check = [
    0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.50
]

rows = []

for threshold in thresholds_to_check:
    pred = (lr_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, pred
        ),
        "predicted_defaults": int(pred.sum()),
    })

threshold_df = pd.DataFrame(rows)

print("LOGISTIC REGRESSION — THRESHOLD COMPARISON")
display(threshold_df.round(4))


# Find validation threshold that maximizes default F1
precision, recall, thresholds = precision_recall_curve(
    y_val, lr_prob
)

f1_values = (
    2 * precision[:-1] * recall[:-1]
    / np.maximum(
        precision[:-1] + recall[:-1],
        1e-12
    )
)

best_index = int(np.argmax(f1_values))
best_threshold = float(thresholds[best_index])

best_pred = (lr_prob >= best_threshold).astype(int)

print("\nBEST VALIDATION F1 THRESHOLD")
print(f"Threshold: {best_threshold:.4f}")
print(f"Precision: {precision_score(y_val, best_pred):.4f}")
print(f"Recall:    {recall_score(y_val, best_pred):.4f}")
print(f"F1:        {f1_score(y_val, best_pred):.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_val, best_pred, labels=[0, 1]))


# Save threshold investigation
threshold_df.to_csv(
    REPORT_DIR / "03_lr_threshold_probe.csv",
    index=False
)

print("\nThreshold investigation saved.")


LOGISTIC REGRESSION — THRESHOLD COMPARISON


,threshold,precision,recall,f1,balanced_accuracy,predicted_defaults
0,0.10,0.1938,0.8402,0.3150,0.6155,57149
1,0.15,0.2358,0.5980,0.3383,0.6301,33424
2,0.20,0.2800,0.3671,0.3177,0.6013,17281
3,0.25,0.3285,0.2000,0.2487,0.5644,8027
4,0.30,0.3672,0.0973,0.1539,0.5340,3494
5,0.35,0.4074,0.0452,0.0814,0.5169,1463
6,0.40,0.4547,0.0206,0.0393,0.5081,596
7,0.50,0.4259,0.0035,0.0069,0.5013,108



BEST VALIDATION F1 THRESHOLD
Threshold: 0.1446
Precision: 0.2321
Recall:    0.6290
F1:        0.3391

Confusion matrix:
[[48194 27433]
 [ 4891  8292]]

Threshold investigation saved.


Logistic Regression assumes the log-odds of default are a linear function of the transformed features. That may be too restrictive for credit-risk relationships.

A Decision Tree can learn nonlinear thresholds and interactions. For example, the effect of high debt-to-income ratio may differ for borrowers with different credit histories.

But Decision Trees have higher variance and can overfit. Therefore, we will constrain their complexity.


## 03.5 — Decision Tree Baseline

### Research question
Can nonlinear decision rules improve loan-default prediction
compared with Logistic Regression?

### Hypothesis
Borrower characteristics may interact nonlinearly.
A Decision Tree may capture these relationships without
explicit polynomial or interaction features.

### Experimental configuration
- Same chronological training and validation periods.
- Same 109 transformed baseline features.
- Median imputation; categorical one-hot encoding.
- No numerical scaling.
- Maximum depth: 12.
- Minimum leaf size: 100.
- No class weighting or threshold tuning initially.

### Evaluation
Compare PR-AUC, ROC-AUC, default precision, recall,
F1, balanced accuracy and confusion matrix.

Record training time and actual tree complexity.

### Decision criteria
If the Decision Tree performs poorly, investigate
whether high variance or unstable probability estimates
justify moving to a Random Forest ensemble.


In [7]:

# 03.5 — Decision Tree Baseline

from time import perf_counter

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

# 1. Tree-specific preprocessing
tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_input_features,
        ),
        (
            "indicators",
            "passthrough",
            indicator_features,
        ),
        (
            "categorical",
            clone(categorical_pipeline),
            categorical_features,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# 2. Constrained Decision Tree
tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=100,
        class_weight=None,
        random_state=42,
    )),
])

# 3. Train on 2013–September 2015
start = perf_counter()
tree_pipeline.fit(X_train, y_train)
tree_seconds = perf_counter() - start

# 4. Validate on October–December 2015
tree_prob = tree_pipeline.predict_proba(X_val)[:, 1]
tree_pred = tree_pipeline.predict(X_val)

assert np.isfinite(tree_prob).all()

# 5. Evaluation at threshold 0.50
tree_result = {
    "model": "Decision Tree",
    "configuration": "Depth=12; leaf=100; unweighted",
    "accuracy": accuracy_score(y_val, tree_pred),
    "balanced_accuracy": balanced_accuracy_score(
        y_val, tree_pred
    ),
    "precision_default": precision_score(
        y_val, tree_pred, zero_division=0
    ),
    "recall_default": recall_score(
        y_val, tree_pred, zero_division=0
    ),
    "f1_default": f1_score(
        y_val, tree_pred, zero_division=0
    ),
    "roc_auc": roc_auc_score(y_val, tree_prob),
    "pr_auc": average_precision_score(y_val, tree_prob),
}

# 6. Update cumulative results without duplicates
results = [
    r for r in results
    if r["model"] != "Decision Tree"
]
results.append(tree_result)

results_df = pd.DataFrame(results)

print("MODEL COMPARISON")
display(results_df.round(4))

print("\nDECISION TREE CONFUSION MATRIX")
print(confusion_matrix(y_val, tree_pred, labels=[0, 1]))

# 7. Model complexity
tree_model = tree_pipeline.named_steps["classifier"]

print("\nMODEL COMPLEXITY")
print("Maximum allowed depth: 12")
print("Actual depth:", tree_model.get_depth())
print("Leaves:", tree_model.get_n_leaves())
print("Transformed features:", tree_model.n_features_in_)
print("Training time:", round(tree_seconds, 2), "seconds")

# 8. Save reproducible experiment results
REPORT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    REPORT_DIR / "03_baseline_comparison.csv",
    index=False,
)

print("\nResults saved.")
print("2016 holdout remains untouched.")


MODEL COMPARISON


,model,configuration,accuracy,balanced_accuracy,precision_default,recall_default,f1_default,roc_auc,pr_auc
0,Dummy (prior),No borrower features,0.8516,0.5000,0.0000,0.0000,0.0000,0.5000,0.1484
1,Logistic Regression,Original features; L2; unweighted,0.8514,0.5013,0.4259,0.0035,0.0069,0.6828,0.2642
2,Decision Tree,Depth=12; leaf=100; unweighted,0.8509,0.5019,0.3561,0.0055,0.0109,0.6416,0.2277



DECISION TREE CONFUSION MATRIX
[[75495   132]
 [13110    73]]

MODEL COMPLEXITY
Maximum allowed depth: 12
Actual depth: 12
Leaves: 1399
Transformed features: 109
Training time: 13.85 seconds

Results saved.
2016 holdout remains untouched.



## 03.6 — Random Forest Baseline

### Research question
Can an ensemble of decision trees improve validation
performance compared with a single Decision Tree?

### Hypothesis
Averaging bootstrap-trained trees with randomized feature
subsets can reduce variance and improve generalization.

### Experimental design
Compare two Random Forest configurations:

1. Standard RF: no class weighting.
2. Balanced RF: balanced_subsample class weighting.

Shared settings:
- 120 trees
- Maximum depth: 14
- Minimum samples per leaf: 50
- Bootstrap sampling: 80%
- Feature sampling: sqrt
- No numerical scaling
- Original 109 transformed features

### Evaluation
Compare ROC-AUC and PR-AUC first.
Report precision, recall and F1 at threshold 0.50.
Inspect exploratory validation-optimal F1 thresholds.

### Decision
Retain the promising RF configuration only if results
justify its additional complexity compared with LR,
Decision Tree and XGBoost.


In [8]:

# 03.6 — Random Forest: Standard vs Balanced

import time
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
)

# Store fitted models and probabilities for later experiments
rf_models = {}
rf_probs = {}
rf_experiments = []

configurations = {
    "Random Forest": None,
    "Random Forest Balanced": "balanced_subsample",
}

for model_name, class_weight in configurations.items():

    print(f"\nTRAINING: {model_name}")

    rf_pipeline = Pipeline([
        (
            "preprocessor",
            clone(tree_pipeline.named_steps["preprocessor"])
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=120,
                max_depth=14,
                min_samples_leaf=50,
                max_features="sqrt",
                class_weight=class_weight,
                bootstrap=True,
                max_samples=0.80,
                random_state=42,
                n_jobs=4,
            )
        ),
    ])

    start = time.perf_counter()
    rf_pipeline.fit(X_train, y_train)
    seconds = time.perf_counter() - start

    # External validation, never used for fitting
    prob = rf_pipeline.predict_proba(X_val)[:, 1]
    pred = (prob >= 0.50).astype(int)

    assert np.isfinite(prob).all()

    result = {
        "model": model_name,
        "configuration": (
            "120 trees; depth=14; leaf=50; "
            f"weight={class_weight}"
        ),
        "accuracy": accuracy_score(y_val, pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_val, pred
        ),
        "precision_default": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall_default": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1_default": f1_score(
            y_val, pred, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_val, prob),
        "pr_auc": average_precision_score(y_val, prob),
    }

    rf_models[model_name] = rf_pipeline
    rf_probs[model_name] = prob
    rf_experiments.append(result)

    # Exploratory F1-maximizing validation threshold
    precision, recall, thresholds = precision_recall_curve(
        y_val, prob
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1], 1e-12
        )
    )

    best_idx = int(np.argmax(f1_values))
    best_threshold = float(thresholds[best_idx])

    best_pred = (prob >= best_threshold).astype(int)

    print("ROC-AUC:", round(result["roc_auc"], 4))
    print("PR-AUC:", round(result["pr_auc"], 4))

    print("\nAt threshold 0.50:")
    print("Precision:", round(result["precision_default"], 4))
    print("Recall:", round(result["recall_default"], 4))
    print("F1:", round(result["f1_default"], 4))

    print("\nExploratory best-F1 threshold:")
    print("Threshold:", round(best_threshold, 4))
    print("Precision:", round(
        precision_score(y_val, best_pred, zero_division=0), 4
    ))
    print("Recall:", round(recall_score(y_val, best_pred), 4))
    print("F1:", round(f1_score(y_val, best_pred), 4))

    print("Confusion matrix:")
    print(confusion_matrix(y_val, best_pred, labels=[0, 1]))

    print("Training seconds:", round(seconds, 2))


# Cumulative comparison: retain existing models
rf_names = set(configurations)

results = [
    r for r in results
    if r["model"] not in rf_names
]

results.extend(rf_experiments)

results_df = pd.DataFrame(results)

print("\nALL MODEL RESULTS — THRESHOLD 0.50")
display(
    results_df[
        [
            "model",
            "roc_auc",
            "pr_auc",
            "precision_default",
            "recall_default",
            "f1_default",
            "balanced_accuracy",
        ]
    ].round(4)
)

# Save reproducible comparison
results_df.to_csv(
    REPORT_DIR / "03_baseline_comparison.csv",
    index=False
)

print("\nRandom Forest experiments saved.")
print("2016 test set remains untouched.")



TRAINING: Random Forest
ROC-AUC: 0.6847
PR-AUC: 0.2674

At threshold 0.50:
Precision: 0.0
Recall: 0.0
F1: 0.0

Exploratory best-F1 threshold:
Threshold: 0.1554
Precision: 0.2396
Recall: 0.5877
F1: 0.3404
Confusion matrix:
[[51032 24595]
 [ 5435  7748]]
Training seconds: 27.86

TRAINING: Random Forest Balanced
ROC-AUC: 0.681
PR-AUC: 0.2633

At threshold 0.50:
Precision: 0.2403
Recall: 0.5566
F1: 0.3357

Exploratory best-F1 threshold:
Threshold: 0.5016
Precision: 0.2414
Recall: 0.5525
F1: 0.336
Confusion matrix:
[[52744 22883]
 [ 5900  7283]]
Training seconds: 27.36

ALL MODEL RESULTS — THRESHOLD 0.50


,model,roc_auc,pr_auc,precision_default,recall_default,f1_default,balanced_accuracy
0,Dummy (prior),0.5000,0.1484,0.0000,0.0000,0.0000,0.5000
1,Logistic Regression,0.6828,0.2642,0.4259,0.0035,0.0069,0.5013
2,Decision Tree,0.6416,0.2277,0.3561,0.0055,0.0109,0.5019
3,Random Forest,0.6847,0.2674,0.0000,0.0000,0.0000,0.5000
4,Random Forest Balanced,0.6810,0.2633,0.2403,0.5566,0.3357,0.6249



Random Forest experiments saved.
2016 test set remains untouched.



### Random Forest — Experimental Conclusion

Random Forest was introduced to investigate whether
bagging and random feature selection improve the
generalization of an individual Decision Tree.

The standard Random Forest improved PR-AUC from 0.2277
(Decision Tree) to 0.2674 and ROC-AUC from 0.6416
to 0.6847.

The balanced configuration substantially improved
default recall at threshold 0.50 (0.5566), but its
ranking performance was slightly worse than standard RF.

After validation threshold optimization, standard RF
achieved F1 = 0.3404, and balanced RF achieved
F1 = 0.3360.

Conclusion:
- Bagging improved the single-tree baseline.
- Class weighting changed classification behavior but
  did not improve PR-AUC.
- Neither RF configuration exceeded XGBoost's
  PR-AUC of 0.2832 or ROC-AUC of 0.7008.

Decision:
Retain Random Forest as an ensemble benchmark.
Prioritize XGBoost for feature-engineering experiments.
No additional RF tuning is justified yet.


In [12]:

# ============================================================
# 03.7 — XGBoost Feature Engineering Ablation
# Fresh experiment: Original vs Enhanced
# No saved models or tuning files required
# ============================================================

import time
import numpy as np
import pandas as pd
import polars as pl

from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    FunctionTransformer,
    StandardScaler,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. Engineer two additional financial features
# ------------------------------------------------------------

engineered_features = [
    "loan_to_income",
    "credit_history_years",
]


def add_engineered_features(X, raw_lf):
    """Create financial features without changing original data."""

    X_new = X.copy().reset_index(drop=True)

    # Loan-to-income ratio
    loan = pd.to_numeric(X_new["loan_amnt"], errors="coerce")
    income = pd.to_numeric(X_new["annual_inc"], errors="coerce")

    X_new["loan_to_income"] = (
        (loan / income.where(income > 0))
        .replace([np.inf, -np.inf], np.nan)
        .astype(np.float32)
    )

    # Retrieve credit-history and issuance dates
    dates = (
        raw_lf
        .select("issue_date", "earliest_cr_line")
        .collect()
        .to_pandas()
        .reset_index(drop=True)
    )

    assert len(dates) == len(X_new), "Row count mismatch"

    issue = pd.to_datetime(
        dates["issue_date"], errors="coerce"
    )

    earliest = pd.to_datetime(
        dates["earliest_cr_line"].astype("string").str.strip(),
        format="%b-%Y",
        errors="coerce",
    )

    months = (
        (issue.dt.year - earliest.dt.year) * 12
        + issue.dt.month - earliest.dt.month
    )

    X_new["credit_history_years"] = (
        (months / 12)
        .where(months >= 0)
        .astype(np.float32)
        .to_numpy()
    )

    return X_new


# Verify raw and prepared row ordering using existing dates
for raw, prepared, expected_rows in [
    (train_lf, train_prepared_lf, len(X_train)),
    (val_lf, val_prepared_lf, len(X_val)),
]:
    raw_dates = raw.select("issue_date").collect().to_series()
    prepared_dates = (
        prepared.select("issue_date").collect().to_series()
    )

    assert len(raw_dates) == expected_rows
    assert raw_dates.equals(prepared_dates), (
        "Raw and prepared row ordering mismatch"
    )


X_train_eng = add_engineered_features(X_train, train_lf)
X_val_eng = add_engineered_features(X_val, val_lf)

print("Original training:", X_train.shape)
print("Enhanced training:", X_train_eng.shape)

print("\nEngineered feature missing values:")
display(X_train_eng[engineered_features].isna().sum())


# ------------------------------------------------------------
# 2. Independent preprocessing pipelines
# ------------------------------------------------------------

def normalize_categories_xgb(X):
    """Normalize categorical values consistently."""
    X = X.copy()

    for c in X.columns:
        X[c] = X[c].astype("string").str.strip()
        X[c] = X[c].replace({
            "": pd.NA,
            "nan": pd.NA,
            "null": pd.NA,
        })
        X[c] = X[c].astype(object).where(
            X[c].notna(), np.nan
        )

    return X


def make_preprocessor(numeric_cols):
    """Create an unfitted preprocessing configuration."""

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        (
            "normalize",
            FunctionTransformer(
                normalize_categories_xgb,
                validate=False,
                feature_names_out="one-to-one",
            ),
        ),
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                dtype=np.float32,
            ),
        ),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipe, numeric_cols),
            ("indicators", "passthrough", indicator_features),
            ("categorical", categorical_pipe, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


original_preprocessor = make_preprocessor(
    numeric_input_features
)

enhanced_preprocessor = make_preprocessor(
    numeric_input_features + engineered_features
)

# Fit both independently on training only
X_train_original = (
    original_preprocessor
    .fit_transform(X_train)
    .astype(np.float32)
)

X_val_original = (
    original_preprocessor
    .transform(X_val)
    .astype(np.float32)
)

X_train_enhanced = (
    enhanced_preprocessor
    .fit_transform(X_train_eng)
    .astype(np.float32)
)

X_val_enhanced = (
    enhanced_preprocessor
    .transform(X_val_eng)
    .astype(np.float32)
)

assert X_train_original.shape[1] == 109
assert X_train_enhanced.shape[1] == 111

print("\nTransformed columns:")
print("Original:", X_train_original.shape[1])
print("Enhanced:", X_train_enhanced.shape[1])


# ------------------------------------------------------------
# 3. Fixed XGBoost configuration
# ------------------------------------------------------------

# New controlled configuration, not recovered tuning results.
# Both models must use exactly the same hyperparameters.

xgb_params = {
    "n_estimators": 350,
    "max_depth": 4,
    "learning_rate": 0.05,
    "min_child_weight": 10,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_lambda": 5,
    "scale_pos_weight": 1,
    "objective": "binary:logistic",
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "random_state": 42,
    "n_jobs": 4,
}

xgb_model = XGBClassifier(**xgb_params)
xgb_model_eng = XGBClassifier(**xgb_params)


# ------------------------------------------------------------
# 4. Train both models
# ------------------------------------------------------------

start = time.perf_counter()
xgb_model.fit(X_train_original, y_train)
original_seconds = time.perf_counter() - start

xgb_prob = xgb_model.predict_proba(
    X_val_original
)[:, 1]


start = time.perf_counter()
xgb_model_eng.fit(X_train_enhanced, y_train)
enhanced_seconds = time.perf_counter() - start

xgb_prob_eng = xgb_model_eng.predict_proba(
    X_val_enhanced
)[:, 1]

assert np.isfinite(xgb_prob).all()
assert np.isfinite(xgb_prob_eng).all()

print("\nTraining completed.")
print("Original:", round(original_seconds, 2), "seconds")
print("Enhanced:", round(enhanced_seconds, 2), "seconds")


# ------------------------------------------------------------
# 5. Evaluation helpers
# ------------------------------------------------------------

def best_f1_info(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(
        y_true, probabilities
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1], 1e-12
        )
    )

    idx = int(np.argmax(f1_values))

    return float(f1_values[idx]), float(thresholds[idx])


def top10_lift(y_true, probabilities):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    n = int(np.ceil(len(y_true) * 0.10))
    indices = np.argsort(-probabilities, kind="stable")[:n]

    return float(y_true[indices].mean() / y_true.mean())


# ------------------------------------------------------------
# 6. Evaluate original vs enhanced
# ------------------------------------------------------------

rows = []

for name, prob in [
    ("Original XGBoost", xgb_prob),
    ("Enhanced XGBoost", xgb_prob_eng),
]:

    best_f1, threshold = best_f1_info(y_val, prob)

    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_val, prob),
        "pr_auc": average_precision_score(y_val, prob),
        "best_val_f1": best_f1,
        "best_val_threshold": threshold,
        "top10_lift": top10_lift(y_val, prob),
    })

comparison_df = pd.DataFrame(rows)

print("\nXGBOOST FEATURE ENGINEERING COMPARISON")
display(comparison_df.round(4))


# ------------------------------------------------------------
# 7. Calculate metric differences
# ------------------------------------------------------------

metric_cols = [
    "roc_auc",
    "pr_auc",
    "best_val_f1",
    "top10_lift",
]

scores = comparison_df.set_index("model")

delta = (
    scores.loc["Enhanced XGBoost", metric_cols].astype(float)
    - scores.loc["Original XGBoost", metric_cols].astype(float)
).round(4)

print("\nCHANGE: ENHANCED - ORIGINAL")
display(delta.to_frame(name="delta"))


# ------------------------------------------------------------
# 8. Enhanced model classification results
# ------------------------------------------------------------

enhanced_f1, enhanced_threshold = best_f1_info(
    y_val, xgb_prob_eng
)

enhanced_pred = (
    xgb_prob_eng >= enhanced_threshold
).astype(int)

print("\nENHANCED XGBOOST — BEST VALIDATION F1")
print("Threshold:", round(enhanced_threshold, 4))
print(
    "Precision:",
    round(precision_score(y_val, enhanced_pred), 4)
)
print(
    "Recall:",
    round(recall_score(y_val, enhanced_pred), 4)
)
print(
    "F1:",
    round(f1_score(y_val, enhanced_pred), 4)
)

print("\nConfusion matrix:")
print(confusion_matrix(y_val, enhanced_pred, labels=[0, 1]))


# ------------------------------------------------------------
# 9. Save fresh experiment results
# ------------------------------------------------------------

REPORT_DIR.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    REPORT_DIR / "03_xgb_engineered_ablation.csv",
    index=False,
)

# Small prediction artifact for later ensemble experiments
np.savez_compressed(
    REPORT_DIR / "03_xgb_validation_predictions.npz",
    y_val=np.asarray(y_val),
    original=xgb_prob,
    enhanced=xgb_prob_eng,
)

print("\nExperiment saved.")
print("Both models used identical hyperparameters.")
print("2016 test set untouched.")


Original training: (457355, 56)
Enhanced training: (457355, 58)

Engineered feature missing values:


loan_to_income          0
credit_history_years    0
dtype: int64


Transformed columns:
Original: 109
Enhanced: 111

Training completed.
Original: 8.37 seconds
Enhanced: 7.59 seconds

XGBOOST FEATURE ENGINEERING COMPARISON


,model,roc_auc,pr_auc,best_val_f1,best_val_threshold,top10_lift
0,Original XGBoost,0.6993,0.2818,0.3529,0.1600,2.2734
1,Enhanced XGBoost,0.6991,0.2818,0.3507,0.1497,2.2620



CHANGE: ENHANCED - ORIGINAL


,delta
roc_auc,-0.0002
pr_auc,-0.0000
best_val_f1,-0.0022
top10_lift,-0.0114



ENHANCED XGBOOST — BEST VALIDATION F1
Threshold: 0.1497
Precision: 0.2452
Recall: 0.6156
F1: 0.3507

Confusion matrix:
[[50643 24984]
 [ 5067  8116]]

Experiment saved.
Both models used identical hyperparameters.
2016 test set untouched.



## 03.8 — Baseline Model Conclusions

### Findings

1. Dummy accuracy of 85.16% demonstrates why accuracy
   alone is unsuitable for this imbalanced dataset.

2. Logistic Regression provides useful risk ranking
   (ROC-AUC 0.6828, PR-AUC 0.2642), but its default
   threshold detects very few defaulters.

3. Threshold optimization improves Logistic Regression
   F1 to 0.3391 on validation data.

4. Decision Tree performs worse than Logistic Regression
   (ROC-AUC 0.6416, PR-AUC 0.2277).

5. Random Forest improves upon the individual tree.
   Class weighting improves recall at threshold 0.50,
   but not overall ranking performance.

6. XGBoost provides the strongest baseline:
   ROC-AUC 0.7008 and PR-AUC 0.2832.

7. Adding loan-to-income and credit-history age
   produces negligible improvement (PR-AUC +0.0005).

### Model-selection decision

Retain original-feature XGBoost as the leading candidate.

Proceed with targeted feature ablation and model
improvement experiments.

All comparisons use October–December 2015 validation
data. The 2016 holdout remains untouched.


In [13]:

# 03.8 — Save baseline experiment documentation

from pathlib import Path
import pandas as pd

ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

REPORT_DIR = ROOT / "reports/modeling"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

baseline = pd.read_csv(
    REPORT_DIR / "03_baseline_comparison.csv"
)

ablation = pd.read_csv(
    REPORT_DIR / "03_xgb_engineered_ablation.csv"
)

report = """# Baseline Modeling — Experimental Conclusions

## Objective
Compare classical ML algorithms for LendingClub default
prediction on the fixed chronological validation period.

## Evaluation policy
- Train: January 2013–September 2015
- Validation: October–December 2015
- Final test: 2016 (untouched)
- Primary selection metric: PR-AUC
- Secondary: ROC-AUC, F1, recall, risk concentration

## Model comparison

""" + baseline.round(4).to_markdown(index=False) + """

## XGBoost engineered-feature ablation

""" + ablation.round(4).to_markdown(index=False) + """

## Findings and decisions

1. Dummy shows high majority-class accuracy but
   zero default-class detection.

2. Logistic Regression learns predictive ranking
   information, although threshold 0.50 is unsuitable
   for detecting many defaults.

3. Decision Tree underperforms Logistic Regression
   on ranking metrics.

4. Random Forest improves upon the single Decision Tree.

5. Balanced Random Forest improves recall at the
   default threshold but does not improve PR-AUC
   over standard Random Forest.

6. XGBoost has the strongest baseline PR-AUC and ROC-AUC.

7. Two engineered features produce negligible improvement.

## Decision

Carry forward original-feature XGBoost as the leading
candidate for targeted feature experiments.

Retain LR and RF as meaningful comparative baselines.

## Limitations

- Validation-tuned F1 thresholds are exploratory.
- Only resolved, issued 36-month loans are modeled.
- Historical results do not guarantee future performance.
- Excluding lender grade/rate changes the prediction task.
- Final 2016 holdout evaluation has not yet occurred.
"""

output_path = (
    REPORT_DIR / "03_baseline_conclusions.md"
)

output_path.write_text(report, encoding="utf-8")

print("Saved:", output_path)
print("Baseline models:", len(baseline))
print("Feature configurations:", len(ablation))
print("Notebook 03: BASELINE EXPERIMENTS COMPLETE")


Saved: d:\ML_Projects\P2P-Lending-Default-Prediction\reports\modeling\03_baseline_conclusions.md
Baseline models: 5
Feature configurations: 2
Notebook 03: BASELINE EXPERIMENTS COMPLETE


In [17]:

# ============================================================
# 03.9 — Ensemble Feasibility Comparison
# ROC-AUC, PR-AUC, F1, Threshold, Precision, Recall
# No retraining required
# ============================================================

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

# ------------------------------------------------------------
# 1. Collect existing validation probabilities
# ------------------------------------------------------------

lr = np.asarray(lr_prob).ravel()
rf = np.asarray(rf_probs["Random Forest"]).ravel()
xgb = np.asarray(xgb_prob).ravel()
y_true = np.asarray(y_val).ravel()

assert len(lr) == len(rf) == len(xgb) == len(y_true)

# ------------------------------------------------------------
# 2. Construct fixed-weight ensembles
# ------------------------------------------------------------

ensembles = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb,

    "LR + XGBoost (50/50)": (
        0.5 * lr + 0.5 * xgb
    ),

    "RF + XGBoost (50/50)": (
        0.5 * rf + 0.5 * xgb
    ),

    "LR + RF + XGBoost": (
        lr + rf + xgb
    ) / 3,
}

# ------------------------------------------------------------
# 3. Find best F1 threshold
# ------------------------------------------------------------

def find_best_f1_threshold(y_true, probabilities):
    """Find F1-maximizing threshold on validation data."""

    precision, recall, thresholds = precision_recall_curve(
        y_true, probabilities
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12
        )
    )

    best_idx = int(np.argmax(f1_values))

    return float(thresholds[best_idx])


# ------------------------------------------------------------
# 4. Evaluate individual models and ensembles
# ------------------------------------------------------------

rows = []

for name, probabilities in ensembles.items():

    # Standard threshold
    predictions_05 = (
        probabilities >= 0.50
    ).astype(int)

    # Best exploratory validation threshold
    best_threshold = find_best_f1_threshold(
        y_true, probabilities
    )

    predictions_best = (
        probabilities >= best_threshold
    ).astype(int)

    rows.append({
        "model": name,

        "roc_auc": roc_auc_score(
            y_true, probabilities
        ),

        "pr_auc": average_precision_score(
            y_true, probabilities
        ),

        "f1_at_0.5": f1_score(
            y_true, predictions_05,
            zero_division=0
        ),

        "best_threshold": best_threshold,

        "best_f1": f1_score(
            y_true, predictions_best,
            zero_division=0
        ),

        "precision": precision_score(
            y_true, predictions_best,
            zero_division=0
        ),

        "recall": recall_score(
            y_true, predictions_best,
            zero_division=0
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions_best
        ),
    })

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

ensemble_comparison = (
    pd.DataFrame(rows)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

print("ENSEMBLE COMPARISON — 2015 VALIDATION")

display(
    ensemble_comparison.round(4)
)

# ------------------------------------------------------------
# 6. Save experiment
# ------------------------------------------------------------

REPORT_DIR.mkdir(parents=True, exist_ok=True)

ensemble_comparison.to_csv(
    REPORT_DIR / "03_ensemble_feasibility.csv",
    index=False
)

print("\nResults saved.")
print("Thresholds optimized on validation only.")
print("2016 test set untouched.")


ENSEMBLE COMPARISON — 2015 VALIDATION


,model,roc_auc,pr_auc,f1_at_0.5,best_threshold,best_f1,precision,recall,balanced_accuracy
0,XGBoost,0.6993,0.2818,0.0056,0.1600,0.3529,0.2549,0.5732,0.6406
1,RF + XGBoost (50/50),0.6960,0.2796,0.0005,0.1576,0.3494,0.2492,0.5842,0.6387
2,LR + RF + XGBoost,0.6951,0.2774,0.0008,0.1575,0.3487,0.2490,0.5818,0.6379
3,LR + XGBoost (50/50),0.6957,0.2773,0.0048,0.1586,0.3486,0.2503,0.5739,0.6372
4,Random Forest,0.6847,0.2674,0.0000,0.1554,0.3404,0.2396,0.5877,0.6313
5,Logistic Regression,0.6828,0.2642,0.0069,0.1446,0.3391,0.2321,0.6290,0.6331



Results saved.
Thresholds optimized on validation only.
2016 test set untouched.


In [18]:

# 03.11 — Inspect PURPOSE and INT_RATE
# Training data only; no validation/test leakage

import pandas as pd
from IPython.display import display

audit = (
    train_lf
    .select(["purpose", "int_rate", "target"])
    .collect()
    .to_pandas()
)

# Clean purpose categories
audit["purpose_clean"] = (
    audit["purpose"]
    .astype("string")
    .str.strip()
    .str.lower()
    .fillna("missing")
    .replace("", "missing")
)

# Parse interest rate as numeric
audit["interest_rate_clean"] = pd.to_numeric(
    audit["int_rate"]
    .astype("string")
    .str.replace("%", "", regex=False)
    .str.strip(),
    errors="coerce"
)

print("PURPOSE DISTRIBUTION AND DEFAULT RATE")

purpose_summary = (
    audit.groupby("purpose_clean", dropna=False)
    .agg(
        loans=("target", "size"),
        defaults=("target", "sum"),
        default_rate=("target", "mean")
    )
    .sort_values("loans", ascending=False)
)

display(purpose_summary.round(4))

print("\nINTEREST RATE SUMMARY")
display(audit["interest_rate_clean"].describe())

print("\nMISSING VALUES")
print(
    audit[["purpose_clean", "interest_rate_clean"]]
    .isna()
    .sum()
)

print("\nUNIQUE PURPOSE CATEGORIES:",
      audit["purpose_clean"].nunique())


PURPOSE DISTRIBUTION AND DEFAULT RATE


,loans,defaults,default_rate
purpose_clean,,,
debt_consolidation,265376,38634,0.1456
credit_card,113659,13256,0.1166
home_improvement,25416,3142,0.1236
other,22793,3740,0.1641
major_purchase,8205,1095,0.1335
medical,4623,792,0.1713
car,4290,510,0.1189
small_business,4272,979,0.2292
moving,3154,632,0.2004



INTEREST RATE SUMMARY


count    457355.0
mean     12.22016
std      3.908938
min          5.32
25%          9.17
50%         12.29
75%         14.65
max         28.99
Name: interest_rate_clean, dtype: Float64


MISSING VALUES
purpose_clean          0
interest_rate_clean    0
dtype: int64

UNIQUE PURPOSE CATEGORIES: 14


In [19]:

# ============================================================
# 03.12 — Purpose and Interest Rate Feature Ablation
# Baseline vs Purpose vs Interest Rate vs Both
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)
from IPython.display import display

# ------------------------------------------------------------
# 1. Load only the two additional columns
# ------------------------------------------------------------

train_extra = (
    train_lf.select(["purpose", "int_rate"])
    .collect().to_pandas().reset_index(drop=True)
)

val_extra = (
    val_lf.select(["purpose", "int_rate"])
    .collect().to_pandas().reset_index(drop=True)
)

assert len(train_extra) == len(X_train)
assert len(val_extra) == len(X_val)

# ------------------------------------------------------------
# 2. Clean purpose using training frequencies only
# ------------------------------------------------------------

def clean_purpose(series):
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace({"": pd.NA})
        .fillna("missing")
    )

train_purpose = clean_purpose(train_extra["purpose"])
val_purpose = clean_purpose(val_extra["purpose"])

common_purposes = set(
    train_purpose.value_counts()
    .loc[lambda counts: counts >= 1000]
    .index
)

train_extra["purpose"] = (
    train_purpose.where(
        train_purpose.isin(common_purposes),
        "rare_other"
    ).astype(object)
)

val_extra["purpose"] = (
    val_purpose.where(
        val_purpose.isin(common_purposes),
        "rare_other"
    ).astype(object)
)

# ------------------------------------------------------------
# 3. Clean interest rate
# ------------------------------------------------------------

for df in [train_extra, val_extra]:
    df["int_rate"] = pd.to_numeric(
        df["int_rate"].astype("string")
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce"
    ).astype(np.float32)

assert train_extra["int_rate"].notna().all()
assert val_extra["int_rate"].notna().all()

# ------------------------------------------------------------
# 4. Add columns without altering baseline data
# ------------------------------------------------------------

X_train_plus = X_train.reset_index(drop=True).copy()
X_val_plus = X_val.reset_index(drop=True).copy()

for col in ["purpose", "int_rate"]:
    X_train_plus[col] = train_extra[col].to_numpy()
    X_val_plus[col] = val_extra[col].to_numpy()

# ------------------------------------------------------------
# 5. Build variants of the original preprocessor
# ------------------------------------------------------------

def make_variant_preprocessor(add_purpose, add_rate):
    pp = clone(original_preprocessor)

    transformers = []

    for name, transformer, columns in pp.transformers:
        columns = list(columns)

        if name == "numeric" and add_rate:
            columns.append("int_rate")

        if name == "categorical" and add_purpose:
            columns.append("purpose")

        transformers.append(
            (name, transformer, columns)
        )

    pp.transformers = transformers
    return pp


def best_f1_and_threshold(y, prob):
    precision, recall, thresholds = (
        precision_recall_curve(y, prob)
    )

    f1 = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12
        )
    )

    idx = int(np.argmax(f1))
    return float(f1[idx]), float(thresholds[idx])


# ------------------------------------------------------------
# 6. Baseline results (already fitted)
# ------------------------------------------------------------

assert len(xgb_prob) == len(y_val)

variants = {
    "Baseline": np.asarray(xgb_prob)
}

experiments = [
    ("Purpose only", True, False),
    ("Interest rate only", False, True),
    ("Purpose + Interest rate", True, True),
]

# ------------------------------------------------------------
# 7. Run three controlled experiments
# ------------------------------------------------------------

for name, add_purpose, add_rate in experiments:

    start = time.perf_counter()

    pp = make_variant_preprocessor(
        add_purpose, add_rate
    )

    Xt = pp.fit_transform(
        X_train_plus
    ).astype(np.float32)

    Xv = pp.transform(
        X_val_plus
    ).astype(np.float32)

    model = clone(xgb_model)
    model.fit(Xt, y_train)

    variants[name] = model.predict_proba(Xv)[:, 1]

    print(
        f"{name}: {Xt.shape[1]} columns, "
        f"{time.perf_counter() - start:.1f}s",
        flush=True
    )

# ------------------------------------------------------------
# 8. Compare metrics
# ------------------------------------------------------------

rows = []

for name, prob in variants.items():
    f1, threshold = best_f1_and_threshold(
        y_val, prob
    )

    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_val, prob),
        "pr_auc": average_precision_score(y_val, prob),
        "best_f1": f1,
        "best_threshold": threshold,
    })

feature_comparison = (
    pd.DataFrame(rows)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

print("\nFEATURE ABLATION RESULTS")
display(feature_comparison.round(4))

# ------------------------------------------------------------
# 9. Save results and validation predictions
# ------------------------------------------------------------

REPORT_DIR.mkdir(parents=True, exist_ok=True)

feature_comparison.to_csv(
    REPORT_DIR / "03_purpose_interest_ablation.csv",
    index=False
)

np.savez_compressed(
    REPORT_DIR / "03_purpose_interest_predictions.npz",
    y_val=np.asarray(y_val),
    **{
        name.replace(" ", "_").replace("+", "and"): prob
        for name, prob in variants.items()
    }
)

print("\nResults and predictions saved.")
print("2016 test set untouched.")


Purpose only: 121 columns, 13.1s
Interest rate only: 110 columns, 12.6s
Purpose + Interest rate: 122 columns, 13.7s

FEATURE ABLATION RESULTS


,model,roc_auc,pr_auc,best_f1,best_threshold
0,Purpose + Interest rate,0.7133,0.2980,0.3633,0.1501
1,Interest rate only,0.7136,0.2974,0.3639,0.1430
2,Purpose only,0.7011,0.2844,0.3547,0.1637
3,Baseline,0.6993,0.2818,0.3529,0.1600



Results and predictions saved.
2016 test set untouched.


In [20]:

# ============================================================
# 03.13 — Grade / Subgrade / Interest Rate Ablation
#
# Compare:
# 1. Baseline                         (existing)
# 2. Interest rate only               (existing)
# 3. Grade only                       (new)
# 4. Subgrade only                    (new)
# 5. Interest rate + Grade            (new)
# 6. Interest rate + Subgrade         (new)
#
# Same XGBoost hyperparameters; no 2016 test access.
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)
from IPython.display import display


# ------------------------------------------------------------
# 1. Check existing experiment variables
# ------------------------------------------------------------

assert "original_preprocessor" in globals()
assert "xgb_model" in globals()
assert "variants" in globals()

assert "Baseline" in variants
assert "Interest rate only" in variants

assert {"grade", "sub_grade"}.issubset(
    set(train_lf.collect_schema().names())
)

# ------------------------------------------------------------
# 2. Load lender-assigned grade features
# ------------------------------------------------------------

train_grades = (
    train_lf
    .select(["target", "grade", "sub_grade"])
    .collect()
    .to_pandas()
    .reset_index(drop=True)
)

val_grades = (
    val_lf
    .select(["target", "grade", "sub_grade"])
    .collect()
    .to_pandas()
    .reset_index(drop=True)
)

# Check row ordering against existing target arrays
assert np.array_equal(
    train_grades["target"].to_numpy(), y_train
)
assert np.array_equal(
    val_grades["target"].to_numpy(), y_val
)

# ------------------------------------------------------------
# 3. Clean categorical values
# ------------------------------------------------------------

def clean_grade(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
        .fillna("MISSING")
        .astype(object)
    )


X_train_grade = X_train_plus.copy().reset_index(drop=True)
X_val_grade = X_val_plus.copy().reset_index(drop=True)

for col in ["grade", "sub_grade"]:
    X_train_grade[col] = clean_grade(
        train_grades[col]
    ).to_numpy()

    X_val_grade[col] = clean_grade(
        val_grades[col]
    ).to_numpy()

print("GRADE FEATURE AUDIT")

for col in ["grade", "sub_grade"]:
    print(
        f"{col}: "
        f"{X_train_grade[col].nunique()} categories, "
        f"{(X_train_grade[col] == 'MISSING').sum()} missing"
    )

# ------------------------------------------------------------
# 4. Create controlled preprocessing variants
# ------------------------------------------------------------

def make_grade_preprocessor(
    include_rate=False,
    grade_column=None
):
    pp = clone(original_preprocessor)

    transformers = []

    for name, transformer, columns in pp.transformers:

        cols = list(columns)

        if name == "numeric" and include_rate:
            cols.append("int_rate")

        if name == "categorical" and grade_column:
            cols.append(grade_column)

        transformers.append(
            (name, transformer, cols)
        )

    pp.transformers = transformers

    return pp


# ------------------------------------------------------------
# 5. Reuse original baseline predictions
# ------------------------------------------------------------

grade_variants = {
    "Baseline": np.asarray(
        variants["Baseline"]
    ),

    "Interest rate only": np.asarray(
        variants["Interest rate only"]
    ),
}

experiments = [
    ("Grade only", False, "grade"),
    ("Subgrade only", False, "sub_grade"),
    ("Interest rate + Grade", True, "grade"),
    ("Interest rate + Subgrade", True, "sub_grade"),
]

# ------------------------------------------------------------
# 6. Train four new models
# ------------------------------------------------------------

for name, include_rate, grade_column in experiments:

    start = time.perf_counter()

    pp = make_grade_preprocessor(
        include_rate=include_rate,
        grade_column=grade_column,
    )

    Xtr = pp.fit_transform(
        X_train_grade
    ).astype(np.float32)

    Xvl = pp.transform(
        X_val_grade
    ).astype(np.float32)

    model = clone(xgb_model)

    model.fit(Xtr, y_train)

    prob = model.predict_proba(Xvl)[:, 1]

    assert np.isfinite(prob).all()

    grade_variants[name] = prob

    print(
        f"{name}: "
        f"{Xtr.shape[1]} columns, "
        f"{time.perf_counter() - start:.1f}s",
        flush=True
    )


# ------------------------------------------------------------
# 7. Evaluation functions
# ------------------------------------------------------------

def best_f1_grade(y, prob):

    precision, recall, thresholds = (
        precision_recall_curve(y, prob)
    )

    f1 = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12
        )
    )

    idx = int(np.argmax(f1))

    return (
        float(f1[idx]),
        float(thresholds[idx])
    )


# ------------------------------------------------------------
# 8. Compare all six configurations
# ------------------------------------------------------------

rows = []

for name, prob in grade_variants.items():

    f1, threshold = best_f1_grade(
        y_val, prob
    )

    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(
            y_val, prob
        ),
        "pr_auc": average_precision_score(
            y_val, prob
        ),
        "best_f1": f1,
        "best_threshold": threshold,
    })

grade_comparison = (
    pd.DataFrame(rows)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

print("\nGRADE AND INTEREST RATE COMPARISON")
display(grade_comparison.round(4))


# ------------------------------------------------------------
# 9. Improvement over interest-rate model
# ------------------------------------------------------------

reference = (
    grade_comparison
    .set_index("model")
    .loc["Interest rate only"]
)

delta_table = grade_comparison.copy()

for metric in ["roc_auc", "pr_auc", "best_f1"]:
    delta_table[f"delta_{metric}"] = (
        delta_table[metric] - reference[metric]
    )

print("\nIMPROVEMENT OVER INTEREST RATE ONLY")

display(
    delta_table[
        [
            "model",
            "delta_roc_auc",
            "delta_pr_auc",
            "delta_best_f1",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 10. Save experiment
# ------------------------------------------------------------

REPORT_DIR.mkdir(parents=True, exist_ok=True)

grade_comparison.to_csv(
    REPORT_DIR / "03_grade_interest_ablation.csv",
    index=False
)

np.savez_compressed(
    REPORT_DIR / "03_grade_interest_predictions.npz",
    y_val=np.asarray(y_val),
    **{
        name.lower()
        .replace(" + ", "_and_")
        .replace(" ", "_"): prob

        for name, prob in grade_variants.items()
    }
)

print("\nExperiment completed and saved.")
print("2016 test set untouched.")


GRADE FEATURE AUDIT
grade: 7 categories, 0 missing
sub_grade: 35 categories, 0 missing
Grade only: 116 columns, 16.0s
Subgrade only: 144 columns, 19.8s
Interest rate + Grade: 117 columns, 12.7s
Interest rate + Subgrade: 145 columns, 18.8s

GRADE AND INTEREST RATE COMPARISON


,model,roc_auc,pr_auc,best_f1,best_threshold
0,Interest rate only,0.7136,0.2974,0.3639,0.1430
1,Interest rate + Subgrade,0.7134,0.2973,0.3637,0.1551
2,Grade only,0.7129,0.2972,0.3651,0.1598
3,Interest rate + Grade,0.7130,0.2971,0.3642,0.1536
4,Subgrade only,0.7103,0.2913,0.3605,0.1591
5,Baseline,0.6993,0.2818,0.3529,0.1600



IMPROVEMENT OVER INTEREST RATE ONLY


,model,delta_roc_auc,delta_pr_auc,delta_best_f1
0,Interest rate only,0.0000,0.0000,0.0000
1,Interest rate + Subgrade,-0.0002,-0.0002,-0.0002
2,Grade only,-0.0006,-0.0002,0.0012
3,Interest rate + Grade,-0.0006,-0.0004,0.0003
4,Subgrade only,-0.0033,-0.0062,-0.0034
5,Baseline,-0.0142,-0.0156,-0.0110



Experiment completed and saved.
2016 test set untouched.


In [21]:

# ============================================================
# 03.14 — Chronological Stacking Ensemble
# Features: Original + Purpose + Interest Rate
# Base models: LR, RF, XGBoost
# Meta-model: Logistic Regression
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)
from IPython.display import display


# ------------------------------------------------------------
# 1. Verify existing data
# ------------------------------------------------------------

assert "X_train_plus" in globals()
assert "X_val_plus" in globals()
assert "original_preprocessor" in globals()
assert "xgb_params" in globals()
assert "variants" in globals()

assert "purpose" in X_train_plus.columns
assert "int_rate" in X_train_plus.columns

assert len(X_train_plus) == len(y_train)
assert len(X_val_plus) == len(y_val)

# Read dates from TRAIN only
train_dates_df = (
    train_lf
    .select(["issue_date", "target"])
    .collect()
    .to_pandas()
)

assert np.array_equal(
    train_dates_df["target"].to_numpy(),
    np.asarray(y_train)
), "Training target alignment mismatch"

issue_dates = pd.to_datetime(
    train_dates_df["issue_date"]
).reset_index(drop=True)

y_array = np.asarray(y_train).ravel()
y_validation = np.asarray(y_val).ravel()


# ------------------------------------------------------------
# 2. Create preprocessor with Purpose + Interest Rate
# ------------------------------------------------------------

def make_stacking_preprocessor():

    pp = clone(original_preprocessor)

    new_transformers = []

    for name, transformer, columns in pp.transformers:

        cols = list(columns)

        if name == "numeric":
            cols.append("int_rate")

        elif name == "categorical":
            cols.append("purpose")

        new_transformers.append(
            (name, transformer, cols)
        )

    pp.transformers = new_transformers

    return pp


# ------------------------------------------------------------
# 3. Create base learners
# ------------------------------------------------------------

def make_stacking_models():

    return {
        "LR": LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),

        "RF": RandomForestClassifier(
            n_estimators=120,
            max_depth=14,
            min_samples_leaf=50,
            max_samples=0.80,
            max_features="sqrt",
            n_jobs=4,
            random_state=42,
        ),

        "XGB": XGBClassifier(**xgb_params),
    }


model_names = ["LR", "RF", "XGB"]


# ------------------------------------------------------------
# 4. Define forward chronological folds
# ------------------------------------------------------------

folds = [
    (
        "2014 H1",
        "2014-01-01",
        "2014-07-01",
    ),
    (
        "2014 H2",
        "2014-07-01",
        "2015-01-01",
    ),
    (
        "2015 Jan-Sep",
        "2015-01-01",
        "2015-10-01",
    ),
]

oof_predictions = np.full(
    (len(y_array), len(model_names)),
    np.nan,
    dtype=np.float32,
)

start_total = time.perf_counter()


# ------------------------------------------------------------
# 5. Generate out-of-fold predictions
# ------------------------------------------------------------

for fold_name, start_date, end_date in folds:

    start_time = time.perf_counter()

    # Only earlier loans can train each fold
    fit_idx = np.flatnonzero(
        (issue_dates < pd.Timestamp(start_date)).to_numpy()
    )

    holdout_idx = np.flatnonzero(
        (
            (issue_dates >= pd.Timestamp(start_date))
            &
            (issue_dates < pd.Timestamp(end_date))
        ).to_numpy()
    )

    assert len(fit_idx) > 0
    assert len(holdout_idx) > 0

    X_fit = X_train_plus.iloc[fit_idx]
    X_holdout = X_train_plus.iloc[holdout_idx]

    y_fit = y_array[fit_idx]

    # Fit preprocessing only on earlier records
    preprocessor = make_stacking_preprocessor()

    X_fit_trans = preprocessor.fit_transform(
        X_fit
    ).astype(np.float32)

    X_holdout_trans = preprocessor.transform(
        X_holdout
    ).astype(np.float32)

    learners = make_stacking_models()

    for j, name in enumerate(model_names):

        learners[name].fit(
            X_fit_trans,
            y_fit,
        )

        oof_predictions[
            holdout_idx, j
        ] = learners[name].predict_proba(
            X_holdout_trans
        )[:, 1]

    print(
        f"{fold_name}: "
        f"train={len(fit_idx):,}, "
        f"OOF={len(holdout_idx):,}, "
        f"time={time.perf_counter()-start_time:.1f}s",
        flush=True,
    )


# ------------------------------------------------------------
# 6. Train meta-model using OOF predictions
# ------------------------------------------------------------

oof_mask = np.isfinite(
    oof_predictions
).all(axis=1)

assert oof_mask.sum() > 0

meta_X = oof_predictions[oof_mask]
meta_y = y_array[oof_mask]

assert np.isfinite(meta_X).all()

meta_model = LogisticRegression(
    max_iter=500,
    random_state=42,
)

meta_model.fit(meta_X, meta_y)

print(
    "\nMeta-model trained on",
    len(meta_y),
    "chronological OOF predictions."
)


# ------------------------------------------------------------
# 7. Refit base learners on full TRAIN period
# ------------------------------------------------------------

start_time = time.perf_counter()

stack_preprocessor = make_stacking_preprocessor()

X_full_trans = stack_preprocessor.fit_transform(
    X_train_plus
).astype(np.float32)

X_validation_trans = stack_preprocessor.transform(
    X_val_plus
).astype(np.float32)

stack_base_models = make_stacking_models()

validation_meta_features = np.zeros(
    (len(y_validation), len(model_names)),
    dtype=np.float32,
)

for j, name in enumerate(model_names):

    stack_base_models[name].fit(
        X_full_trans,
        y_array,
    )

    validation_meta_features[:, j] = (
        stack_base_models[name]
        .predict_proba(X_validation_trans)[:, 1]
    )

print(
    "Full-data base models trained in",
    round(time.perf_counter()-start_time, 1),
    "seconds."
)


# ------------------------------------------------------------
# 8. Final stacking predictions
# ------------------------------------------------------------

stack_prob = meta_model.predict_proba(
    validation_meta_features
)[:, 1]

assert np.isfinite(stack_prob).all()

# Compare with previously tested origination models
comparison_predictions = {
    "Interest rate only":
        np.asarray(variants["Interest rate only"]),

    "Purpose + Interest rate XGB":
        validation_meta_features[:, 2],

    "Stacking LR + RF + XGB":
        stack_prob,
}


# ------------------------------------------------------------
# 9. Evaluate all models
# ------------------------------------------------------------

def evaluate_stack_predictions(name, prob):

    precision, recall, thresholds = (
        precision_recall_curve(
            y_validation,
            prob,
        )
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / np.maximum(
            precision[:-1] + recall[:-1],
            1e-12,
        )
    )

    idx = int(np.argmax(f1_values))
    threshold = float(thresholds[idx])

    pred = (prob >= threshold).astype(int)

    return {
        "model": name,

        "roc_auc": roc_auc_score(
            y_validation, prob
        ),

        "pr_auc": average_precision_score(
            y_validation, prob
        ),

        "best_f1": f1_score(
            y_validation, pred
        ),

        "threshold": threshold,

        "precision": precision_score(
            y_validation, pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_validation, pred,
            zero_division=0
        ),
    }


stack_results = pd.DataFrame([
    evaluate_stack_predictions(name, prob)
    for name, prob in comparison_predictions.items()
])

stack_results = (
    stack_results
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

print("\nSTACKING ENSEMBLE COMPARISON")
display(stack_results.round(4))

print(
    "\nTotal time:",
    round(time.perf_counter() - start_total, 1),
    "seconds"
)


# ------------------------------------------------------------
# 10. Save experiment
# ------------------------------------------------------------

REPORT_DIR.mkdir(parents=True, exist_ok=True)

stack_results.to_csv(
    REPORT_DIR / "03_stacking_comparison.csv",
    index=False,
)

np.savez_compressed(
    REPORT_DIR / "03_stacking_validation_predictions.npz",
    y_val=y_validation,
    xgb=validation_meta_features[:, 2],
    stacking=stack_prob,
    interest_rate_only=np.asarray(
        variants["Interest rate only"]
    ),
)

print("\nStacking experiment complete.")
print("2016 test set untouched.")


2014 H1: train=100,422, OOF=71,955, time=7.0s
2014 H2: train=172,377, OOF=90,615, time=13.9s
2015 Jan-Sep: train=262,992, OOF=194,363, time=23.3s

Meta-model trained on 356933 chronological OOF predictions.
Full-data base models trained in 40.6 seconds.

STACKING ENSEMBLE COMPARISON


,model,roc_auc,pr_auc,best_f1,threshold,precision,recall
0,Purpose + Interest rate XGB,0.7133,0.2980,0.3633,0.1501,0.2645,0.5799
1,Interest rate only,0.7136,0.2974,0.3639,0.1430,0.2588,0.6123
2,Stacking LR + RF + XGB,0.7114,0.2956,0.3627,0.1660,0.2707,0.5493



Total time: 85.1 seconds

Stacking experiment complete.
2016 test set untouched.


In [23]:

# ============================================================
# 03.15 — Save Selected XGBoost Candidate
# Purpose + Interest Rate
# No retraining; 2016 test remains untouched
# ============================================================

import cloudpickle
import json
import numpy as np

from pathlib import Path

MODEL_DIR = ROOT / "saved_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Already fitted by chronological stacking experiment
final_preprocessor = stack_preprocessor
final_model = stack_base_models["XGB"]

# Use the actual validation-selected threshold
selected_row = stack_results.loc[
    stack_results["model"] == "Purpose + Interest rate XGB"
].iloc[0]

selected_threshold = float(selected_row["threshold"])

bundle = {
    "preprocessor": final_preprocessor,
    "model": final_model,
    "input_columns": list(X_train_plus.columns),
    "threshold": selected_threshold,
    "model_name": "XGBoost",
    "prediction_stage": "loan_origination",
    "extra_features": ["purpose", "int_rate"],
    "training_period": "2013-01 to 2015-09",
    "validation_period": "2015-10 to 2015-12",
}

model_path = (
    MODEL_DIR / "03_origination_xgb_candidate.pkl"
)

with open(model_path, "wb") as f:
    cloudpickle.dump(bundle, f)

# Immediately test saved artifact
with open(model_path, "rb") as f:
    loaded = cloudpickle.load(f)

sample = X_val_plus.iloc[:100].copy()

before = final_model.predict_proba(
    final_preprocessor.transform(sample).astype(np.float32)
)[:, 1]

after = loaded["model"].predict_proba(
    loaded["preprocessor"].transform(sample).astype(np.float32)
)[:, 1]

np.testing.assert_allclose(
    before, after, rtol=1e-6, atol=1e-7
)

print("MODEL SAVED:", model_path)
print("Serialization verification: PASS")
print("Threshold:", round(selected_threshold, 4))
print("Input columns:", len(bundle["input_columns"]))
print("Encoded columns:", final_preprocessor.transform(sample).shape[1])
print("2016 test set untouched.")


MODEL SAVED: d:\ML_Projects\P2P-Lending-Default-Prediction\saved_models\03_origination_xgb_candidate.pkl
Serialization verification: PASS
Threshold: 0.1501
Input columns: 58
Encoded columns: 122
2016 test set untouched.


After doing baseline comparesions we are retaining the interest rate and purpose , but excluding the engineered feature as they give negligible performance addition.